# Train HeadScan-Lite from request-free MakeHuman data in Google Colab

Use a GPU runtime and choose **Runtime → Run all**. No dataset request or manual download is required.

The notebook now builds its training set automatically from MakeHuman's official **core base mesh + morph targets**, which MakeHuman documents as **CC0**. It does not import MakeHuman application code and does not use third-party community assets.

### What happens automatically

1. Mount Google Drive.
2. Clone the current `Head_model` repo.
3. Restore a prepared synthetic dataset cache if one exists.
4. Otherwise sparse-clone only `makehuman/data/3dobjs` and `makehuman/data/targets` from the official MakeHuman repo.
5. Verify the MakeHuman repository license text contains the expected CC0 core-asset statement.
6. Generate **2,500 registered identities × 8 camera views** by default.
7. Randomize head/face morph targets, macro identity shape, camera pose, skin tone, lighting, background, hair occlusion, blur, sensor noise and JPEG quality.
8. Save exact registered head vertices and foreground masks.
9. Cache the generated dataset in Drive so later free-Colab sessions do not regenerate it.
10. Train/resume HeadScan-Lite and export ONNX + fixed triangle topology.

The default cache is named from the generation settings, for example `MyDrive/head_model/makehuman_synth_v1_2500ids_8views_320px.zip`. Checkpoints go to `MyDrive/head_model/headscan-lite-makehuman-2500/`.

Default training is 256×256, 35 epochs, batch size 2, 1–8 randomly sampled views, up to 128 PCA components, MobileNetV3-Small and a 3-layer/4-head variable-view Transformer. The MobileNet backbone trains from scratch by default so the training path does not depend on ImageNet-derived pretrained weights.

`head_faces.npy` is packaged next to `headscan_lite.onnx`; the phone combines those fixed triangle indices with the predicted `vertices_mm` output to construct the final mesh.

This is **synthetic pretraining**. It removes the access/licensing blocker and gives perfect topology/geometry supervision, but real-world scan accuracy still needs to be measured on a separate real-head benchmark.

In [ ]:
import pathlib, shutil, subprocess
bootstrap = pathlib.Path('/content/head_model_bootstrap')
if bootstrap.exists():
    shutil.rmtree(bootstrap)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Persie0/Head_model.git', str(bootstrap)], check=True)
subprocess.run(['git', '-C', str(bootstrap), 'rev-parse', '--short', 'HEAD'], check=True)
script = bootstrap / 'colab' / 'train_headscan_lite_colab.py'
source = script.read_text(encoding='utf-8')
required = ['HEADSCAN_COLAB_VERSION', 'v3-makehuman', 'SYNTH_IDENTITIES', 'clone_makehuman_cc0_assets', 'run_with_heartbeat']
if not all(token in source for token in required):
    raise RuntimeError('Fresh clone does not contain the expected current MakeHuman Colab trainer. Stop instead of running stale code.')
exec(compile(source, str(script), 'exec'), {'__name__': '__main__'})
